# Cleaning

In this script, we clean the speech data.

The data were obtained from https://www.bundestag.de/services/opendata using the API. They are already structured and enriched with additional master data information on the speakers.

In this script, we clean the data.

## Preparation

First, we instal and load the libraries needed in this script.

In [1]:
!pip install clean-text

In [2]:
!pip install unidecode

In [3]:
import pandas as pd
import numpy as np
import os
from collections import Counter
from nltk import ngrams
from functools import reduce
import string
from cleantext import clean

Then, we mount the drive and load the data:

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
path_to_data = '/content/drive/MyDrive/TL/data/'

# load speech data:
df = pd.read_parquet(path_to_data + 'speeches.parquet')

## Explore data

In [6]:
df.head() # first five rows

,legislative_period,date,speaker_id,first_name,last_name,role,party,speech
0,19,15.01.2020,11004699,Astrid,Damerow,keine,CDU/CSU,Frau Präsidentin! Verehrte Kolleginnen und Kol...
1,19,15.01.2020,11004393,Johann,Saathoff,keine,SPD,Sehr geehrte Frau Präsidentin! Liebe Kolleginn...
2,19,15.01.2020,11003706,Artur,Auernhammer,keine,CDU/CSU,Verehrte Frau Präsidentin! Liebe Kolleginnen u...
3,19,15.01.2020,11003604,Friedrich,Ostendorff,keine,Die Grünen,Sehr geehrte Frau Präsidentin! Liebe Kolleginn...
4,19,15.01.2020,11003740,Heidrun,Bluhm-Förster,keine,Die Linke,Frau Präsidentin! Verehrte Kolleginnen und Kol...


In [7]:
df.describe()

,legislative_period,date,speaker_id,first_name,last_name,role,party,speech
count,38891,38891,38891,38891,38891,38891,38891,38891
unique,2,312,1020,509,928,122,6,38828
top,20,21.09.2023,11003231,Michael,Müller,keine,CDU/CSU,Ja.
freq,26151,226,471,784,520,33778,10083,26


## Removing special characters

First, we assess which special characters appear in the speeches and at what frequency:

In [8]:
all_chars = "".join(df['speech'].tolist())
char_freqs = Counter(all_chars)

special_char_freqs = Counter({char: freq for char, freq in char_freqs.items() if char not in string.ascii_letters})

print(special_char_freqs)

Counter({' ': 19414702, ',': 1338005, '.': 1043422, 'ü': 683023, 'ä': 550371, 'ö': 269925, '–': 175816, ':': 146241, 'ß': 122608, '0': 116790, '-': 114723, '!': 93930, '2': 77044, ';': 61455, '1': 56413, '\n': 55811, '?': 48359, '„': 36842, '“': 36810, '5': 27327, '3': 23832, '4': 18772, '9': 17415, 'Ü': 16520, '6': 15127, '7': 13821, '8': 13125, '\u202f': 11664, 'Ä': 9307, '/': 8923, 'Ö': 7375, '‑': 3396, '§': 2499, '…': 2275, '’': 1042, 'é': 627, ')': 275, 'ʼ': 206, 'à': 193, '&': 178, '‚': 168, '(': 152, '#': 146, 'ğ': 116, 'á': 115, 'è': 90, '+': 89, 'ć': 82, 'É': 61, 'ó': 58, 'í': 54, '*': 52, 'ç': 42, 'å': 25, 'ï': 24, 'č': 22, '"': 21, '[': 16, ']': 16, '_': 15, 'ś': 14, '%': 13, 'ô': 13, 'ł': 13, '‘': 12, 'ê': 10, 'ă': 10, "'": 10, '@': 10, 'ę': 10, 'İ': 10, 'ş': 9, '”': 8, 'ě': 7, 'λ': 6, 'ñ': 6, 'ž': 6, 'ā': 6, 'ē': 6, 'ã': 5, 'ń': 5, 'Č': 5, 'î': 5, 'š': 5, 'ė': 5, '‒': 4, '=': 4, '·': 4, 'Å': 3, 'ν': 3, 'ο': 3, 'Ş': 3, '\u2001': 3, '€': 2, '°': 2, 'À': 2, 'â': 2, 'ź': 2, 'ό

We replace special characters that have a word as their equivalent:

In [9]:
def clean_special_chars(speech):
  speech.replace('%', ' Prozent')
  speech.replace('§', 'Paragraf')
  return(speech)

In [10]:
df['speech'] = df['speech'].apply(clean_special_chars)

Then, we use the 'clean' function from the clean_text library for further cleaning:

In [11]:
def clean_customized(speech):
  speech = clean(speech,
                 fix_unicode=True,               # fix various unicode errors
                 to_ascii=True,                  # transliterate to closest ASCII representation
                 lower=False,                    # lowercase text
                 no_line_breaks=True,            # fully strip line breaks as opposed to only normalizing them
                 no_urls=False,                  # replace all URLs with a special token
                 no_emails=False,                # replace all email addresses with a special token
                 no_phone_numbers=False,         # replace all phone numbers with a special token
                 no_numbers=False,               # replace all numbers with a special token
                 no_digits=False,                # replace all digits with a special token
                 no_currency_symbols=False,      # replace all currency symbols with a special token
                 no_punct=False,                 # remove punctuations
                 replace_with_punct="",          # instead of removing punctuations you may replace them
                 replace_with_url="",
                 replace_with_email="",
                 replace_with_phone_number="",
                 replace_with_number="",
                 replace_with_digit="0",
                 replace_with_currency_symbol="",
                 lang = "de")
  return(speech)

In [12]:
df['speech'] = df['speech'].apply(clean_customized)

We look at one example:

In [13]:
df.iloc[16]['speech']

'Lieber Kollege Höferlin, vielen Dank für diese Frage. Wenn es Sicherheitslücken in anderen europäischen Ländern gibt, dann sollten wir uns doch gemeinsam bemühen - das ist meine Meinung -, diese zu schließen, statt sie in Deutschland zu übernehmen. Es kann doch nicht unser politisches Handeln sein, dass wir uns daran ein Beispiel nehmen. Das ist auf jeden Fall nicht unser Kurs; das kann ich Ihnen sagen. Ich habe es Ihnen gesagt: Auch die Inhaber von Privatpilotlizenzen haben Zugänge zu allen sicherheitsrelevanten Bereichen. Das, was Sie fordern, ist im Sinne einer Gleichbehandlung völlig absurd, und es ist auch nicht zu Ende gedacht. Deshalb lehnen wir den Antrag der FDP entschieden ab, freuen uns aber trotzdem auf die weiteren Beratungen mit Ihnen. Vielen Dank.'

Then we check how the preprocessing has affected the frequencies of special characters:

In [14]:
all_chars_post = "".join(df['speech'].tolist())
char_freqs_post = Counter(all_chars_post)

special_chars_post_freqs = Counter({char: freq for char, freq in char_freqs_post.items() if char not in string.ascii_letters})

print(special_chars_post_freqs)

Counter({' ': 18387870, ',': 1338005, '.': 1050247, 'ü': 683023, 'ä': 550371, '-': 293942, 'ö': 269925, ':': 146241, 'ß': 122608, '0': 116790, '!': 93931, '2': 77047, '"': 73681, ';': 61455, '1': 56414, '?': 48359, '5': 27327, '3': 23834, '4': 18772, '9': 17415, 'Ü': 16520, '6': 15127, '7': 13821, '8': 13125, 'Ä': 9307, '/': 8925, 'Ö': 7375, "'": 1442, ')': 275, '&': 178, '(': 152, '#': 146, '+': 89, '*': 58, '[': 16, ']': 16, '_': 15, '%': 13, '@': 10, '=': 4})


This looks better than before. We save the cleaned data:

In [15]:
df.to_parquet(path_to_data + 'speeches_clean.parquet')